# Module 05 - Improving RAG

**Duration:** 60 minutes

A basic RAG pipeline works, but it has predictable failure modes.
This module covers four techniques that address the most common ones.
Each technique has a before/after comparison so you can see the difference directly.

---


## Setup


In [ ]:
import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings('ignore', category=TqdmExperimentalWarning)

from ragsst.ragtool import RAGTool
import requests, json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2'

tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.setup_vec_store()
print('Ready.')


In [ ]:
def generate(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': MODEL, 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


def basic_rag(query: str, n: int = 3) -> str:
    context = tool.get_relevant_text(query, nresults=n)
    prompt = tool.get_context_prompt(query, context)
    return generate(prompt)


---

## Technique 1: Re-ranking

Vector search retrieves the top-k most similar chunks by embedding distance.
But embedding similarity is not the same as relevance to a specific question.
A chunk can be topically related without actually answering what was asked.

### Why bi-encoders fail at precision

Embedding models used for retrieval are called **bi-encoders**: they encode
the query and each document *independently* into vectors. This makes them fast
(you can pre-compute document vectors), but the encoding of each piece happens
without any awareness of the other.

A **cross-encoder** solves this by encoding the *pair* (query, document) together.
The model can see both simultaneously and produce a relevance score that captures
the interaction between them. This is much more accurate — but also much slower,
because you cannot pre-compute document representations.

### The retrieve-then-rerank pattern

The standard solution is to combine both:
1. Use the bi-encoder to retrieve the top-N candidates quickly (e.g. N=10 or 20)
2. Pass only the candidates to the cross-encoder for accurate re-scoring
3. Keep the top-k after re-ranking (e.g. k=3)

This way you get the speed of bi-encoders for the bulk of the work,
and the accuracy of cross-encoders for the final ranking.

### Latency budget

Cross-encoding N=10 candidates with a small model takes ~50–100ms.
For most applications this is acceptable. If latency is critical, reduce N.


In [ ]:
from sentence_transformers import CrossEncoder

# This model is specifically trained for passage re-ranking
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def rag_with_reranking(query: str, retrieve_n: int = 10, rerank_to: int = 3) -> str:
    # Step 1: retrieve more candidates than we need
    results = tool.collection.query(query_texts=[query], n_results=retrieve_n)
    candidates = results['documents'][0]

    # Step 2: score each candidate against the query
    pairs = [[query, doc] for doc in candidates]
    scores = reranker.predict(pairs)

    # Step 3: sort by score and keep the top ones
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:rerank_to]]

    context = '\n'.join(top_docs)
    return generate(tool.get_context_prompt(query, context))


query = 'What workshops does the AI service center run on Tuesdays?'

print('Without reranking:')
print(basic_rag(query, n=3))

print('\nWith reranking (retrieve 10, rerank to 3):')
print(rag_with_reranking(query))


The cross-encoder reads the query and each chunk together, so it can catch
cases where the embedding similarity was misleading.

**Exercise:** Find a question where reranking makes no difference.
Then find one where it clearly helps. What characterises the two cases?


---

## Technique 2: HyDE (Hypothetical Document Embeddings)

A question and its answer are often expressed very differently.
*"When does the paper reading session take place?"* does not look much like
*"The paper reading sessions take place every Wednesday afternoon."*

In embedding space, questions and answers occupy different regions. 
The query *"When does X take place?"* is semantically close to other questions about timing,
not to answers about timing.

### The asymmetry between questions and answers

This is actually a known limitation of bi-encoder training. Models like
`multi-qa-mpnet-base-cos-v1` were specifically trained on (question, answer) pairs
to reduce this gap — but it never fully disappears.

HyDE takes a different approach: instead of trying to make the question's embedding
land close to the answer's embedding, it **generates a fake answer** and embeds that.
A fake answer *looks* like an answer, so it should land close to real answers in the
vector space.

### When HyDE helps (and when it doesn't)

**Helps when:**
- Questions are short or abstract
- The document uses very different vocabulary than natural question phrasing
- The model is good enough to generate a plausible fake answer

**Doesn't help (or hurts) when:**
- The LLM hallucinates a fake answer that leads retrieval in the wrong direction
- The question is already well-matched to the document vocabulary
- Latency matters (HyDE requires an extra full LLM call)

> **HyDE is a high-variance technique**: when it works, the improvement is
> dramatic; when it fails, it can be worse than baseline. Always measure
> (Module 06) before deploying it.


In [ ]:
def generate_hypothetical_answer(query: str) -> str:
    prompt = (
        'Write a short, factual passage that would directly answer the following question. '
        'Do not say you are guessing. Just write the passage as if it were from a document.\n\n'
        f'Question: {query}\n'
        'Passage:'
    )
    return generate(prompt, temp=0.5)


def rag_with_hyde(query: str, n: int = 3) -> str:
    # Generate a hypothetical answer
    hypothetical = generate_hypothetical_answer(query)

    # Retrieve using the hypothetical answer instead of the question
    context = tool.get_relevant_text(hypothetical, nresults=n)

    # Answer using the original question and the retrieved context
    return generate(tool.get_context_prompt(query, context))


query = 'When do the paper reading sessions take place?'

print('Hypothetical answer generated by LLM:')
print(generate_hypothetical_answer(query))

print('\nWithout HyDE:')
print(basic_rag(query))

print('\nWith HyDE:')
print(rag_with_hyde(query))


HyDE adds an extra LLM call, which increases latency.
It is most useful when questions are short or under-specified.

**Exercise:** Try HyDE on a question where the basic RAG already works well.
Does HyDE make things worse? Can you find a case where it retrieves the wrong document?


---

## Technique 3: Multi-Query

A single query has a single embedding, which searches a single direction in vector space.
If the relevant document uses slightly different vocabulary, it might not come up.

### The coverage problem

Consider the query: *"What free resources does the AI center provide?"*
This embeds in a region near "free", "resources", "AI center".

A document that says *"The AI Service Center offers complimentary workshops and computing time"*
uses none of those words — but it answers the question perfectly.
The word "complimentary" is closer to "courtesy" in embedding space than to "free".

Multi-query broadens the search by generating several rephrasings and querying for each.

### Why it works

Different phrasings of the same question land in different regions of embedding space.
Some of those regions will be closer to the relevant document.
By querying from multiple directions and taking the union of results,
you dramatically increase the chance of finding the right chunk.

### The duplicate problem

When you merge results from multiple queries, the same chunk may appear multiple times.
Simply deduplicating works, but wastes the information that a chunk was retrieved by
*multiple* queries — which is strong evidence it is relevant. RAG-Fusion (Technique 4)
exploits this signal.


In [ ]:
def generate_query_variants(query: str, n: int = 3) -> list[str]:
    prompt = (
        f'Write {n} different versions of the following question. '
        'Each version should ask for the same information but use different words. '
        'Output only the questions, one per line, with no numbering or extra text.\n\n'
        f'Question: {query}'
    )
    response = generate(prompt, temp=0.7)
    variants = [line.strip() for line in response.strip().split('\n') if line.strip()]
    return variants[:n]


def rag_with_multiquery(query: str, n_variants: int = 3, n_results_each: int = 2) -> str:
    variants = generate_query_variants(query, n_variants)
    all_queries = [query] + variants

    # Retrieve for each query variant and deduplicate
    seen = set()
    all_docs = []
    for q in all_queries:
        results = tool.collection.query(query_texts=[q], n_results=n_results_each)
        for doc in results['documents'][0]:
            if doc not in seen:
                seen.add(doc)
                all_docs.append(doc)

    context = '\n\n'.join(all_docs)
    return generate(tool.get_context_prompt(query, context))


query = 'What free resources does the AI center provide?'

print('Query variants:')
for v in generate_query_variants(query):
    print(' -', v)

print('\nWithout multi-query:')
print(basic_rag(query))

print('\nWith multi-query:')
print(rag_with_multiquery(query))


---

## Technique 4: RAG-Fusion

RAG-Fusion is multi-query + smarter result merging.
Instead of just deduplicating, it uses **Reciprocal Rank Fusion (RRF)** to score
each document based on how highly it ranked across all the queries.

### Reciprocal Rank Fusion formula

For a document *d* appearing at rank *r* in list *l*:

```
RRF_score(d) = Σ  1 / (k + r)
               l
```

The constant *k* (typically 60) prevents top-ranked documents from dominating.

**Why this is better than voting or averaging scores:**
- A document that appears at rank 1 in all three query variants scores very highly
- A document that appears at rank 10 in all three still scores decently  
- The k=60 constant smooths out the very large differences between ranks 1 and 2

RRF was originally developed for combining text search rankings in 2009.
It turns out to work just as well for combining vector search rankings —
and it is extremely simple to implement.

### RRF vs. score normalisation

An alternative is to normalise and average the similarity scores across queries.
RRF tends to outperform score averaging in practice because:
- Different queries produce scores on different scales (hard to normalise)
- Rank is a more robust signal than raw score

### Summary: which technique to use?

The four techniques address different failure modes. In practice, start with
the diagnostic from Module 06 to identify *which* stage is failing before
applying any technique:

| If this is failing | Try this |
|-------------------|----------|
| Right chunks not retrieved at all | Multi-query or HyDE |
| Right chunks retrieved but ranked low | Re-ranking |
| Multiple queries finding same things | RAG-Fusion |
| LLM ignoring context | Prompt engineering (Module 04) |


In [ ]:
def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return [doc for doc, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]


def rag_fusion(query: str, n_variants: int = 3, n_per_query: int = 5, final_n: int = 3) -> str:
    variants = generate_query_variants(query, n_variants)
    all_queries = [query] + variants

    # Get a ranked list of docs for each query
    ranked_lists = []
    for q in all_queries:
        results = tool.collection.query(query_texts=[q], n_results=n_per_query)
        ranked_lists.append(results['documents'][0])

    # Fuse the ranked lists
    fused = reciprocal_rank_fusion(ranked_lists)

    context = '\n\n'.join(fused[:final_n])
    return generate(tool.get_context_prompt(query, context))


query = 'What free resources does the AI center provide?'

print('Without RAG-Fusion:')
print(basic_rag(query))

print('\nWith RAG-Fusion:')
print(rag_fusion(query))


---

## Summary

| Technique | What it fixes | Extra cost | When to reach for it |
| --- | --- | --- | --- |
| **Re-ranking** | Retrieved chunks are topically close but not actually relevant | Cross-encoder call per candidate | Precision is low; right docs retrieved but ranked wrong |
| **HyDE** | Question vocabulary very different from document vocabulary | 1 extra LLM call | Short/abstract queries; multilingual mismatch |
| **Multi-query** | Single query misses relevant docs due to vocabulary mismatch | N extra retrieval calls | Recall is low; you know the answer is in the corpus |
| **RAG-Fusion** | Same as multi-query, with smarter result merging | N extra retrieval calls | Multi-query working but ordering is inconsistent |

None of these is always better. The right choice depends on your documents and query patterns.
That is what Module 06 is about — measuring before deciding.

---

**Exercises**

1. Combine re-ranking with multi-query: retrieve using multiple queries,
   then pass all candidates to the cross-encoder and keep the top 3.
   Does this improve on either technique alone?

2. Try HyDE on a query about Wild Tales. Does the hypothetical answer
   mention anything not in the document? What happens to retrieval if it does?

3. For RAG-Fusion, print the fused ranking alongside the single-query ranking.
   Find a case where a document moves from rank 5 to rank 1 after fusion.
   What does that tell you about that document?

4. *Hard:* Implement a version of HyDE that generates *multiple* hypothetical answers
   and queries for each one (combining HyDE and Multi-Query). Does it outperform either alone?

---

**Further reading**

- HyDE paper: https://arxiv.org/abs/2212.10496
- RAG-Fusion post: https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1
- Cross-encoders vs bi-encoders: https://www.sbert.net/docs/cross_encoder/usage.html
- RRF original paper: https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
